# 1_SynAnalyzer_ConvertImarisStatsFile.ipynb
This notebook reads in Imaris statistics files and generates Python-friendly CSV files 
with the relevent surface statistics and image metadata added. The exact parameters used for the
conversion process were determined empirically and may need adjustment should Imaris alter the format
of their files. 

In [13]:
# Import packages
import numpy as np
import pandas as pd
import glob
import os
import xlrd

In [18]:
#                *** WHAT TO ANALYZE // WHERE TO GET/STORE **
# Key identifiers 
batchID = 'SynAnalyzer_DemoBatch'

# Voxel dimensions for converting XYZs (in um)
voxelWidth = 0.0619
voxelHeight = 0.0619
voxelDepth = 0.34

# Threshold for volume of surfaces to include. Must be greater than zero!
#  Volumes that are too small do not have a mean intensity and it will result in a runtime error
#  The threshold shown below in the demo code was determined empirically
volThresh = 0.1

# Directories 
#   existing ones 
dirMain = '/Users/joyfranco/Partners HealthCare Dropbox/Joy Franco/JF_Shared/Data/WSS/BatchAnalysis/'
dirBA = dirMain+batchID+'/'
dirData = dirBA+'ImarisStatsFiles/'

#   ones that need to be made    
dirSV = dirBA+'XYZCSVs/'        

In [19]:
#           *** INITIALIZE RUN SPECIFIC DIRECTORY ETC FOR STORING RESULTS **
# Create directory for storing spreadsheetS and summary plotS for this run
if not os.path.exists(dirSV): os.mkdir(dirSV)

In [20]:
#           *** GENERATE A LIST OF FILES THAT WILL BE CONVERTED **
# This list is based off the available Excel results. 
# Generate a list of all xls files added by the user
os.chdir(dirData)
files = glob.glob('*Syn.xls')
print(files)

['WSS_031.03.T3.01.Zs.4C.XYZ.PostSyn.xls', 'WSS_031.03.T3.01.Zs.4C.XYZ.PreSyn.xls', 'WSS_030.01.T2.01.Zs.4C.XYZ.PreSyn.xls', 'WSS_030.01.T2.01.Zs.4C.XYZ.PostSyn.xls']


In [21]:
# Iterate through the xls files, extract the relavent sheet, and reformat
for file in files:
    # Read in the sheets that correspond to each desired df
    dfXYZ = pd.read_excel(file,skiprows=1, sheet_name='Position')
    dfVol = pd.read_excel(file,skiprows=1, sheet_name='Volume')
    dfIMO = pd.read_excel(file, sheet_name='Intensity Mean Ch=1 Img=1')
    dfIMT = pd.read_excel(file,sheet_name='Intensity Mean Ch=2 Img=1')
    dfIMTh = pd.read_excel(file,sheet_name='Intensity Mean Ch=3 Img=1')
    dfIMF = pd.read_excel(file,sheet_name='Intensity Mean Ch=4 Img=1')
    
    # Reformat each df to make it a proper df rather than sheet
    dfXYZ = dfXYZ.drop(columns=['Unit', 'Category', 'Collection', 'Time'])
    dfXYZ.set_index('ID', inplace=True)
    dfVol = dfVol.drop(columns=['Unit', 'Category','Time'])
    dfVol.set_index('ID', inplace=True)
    
    #dfIMO = dfIMO.drop(columns=['Unit', 'Category', 'Channel','Image','Time'])
    dfIMO.columns = dfIMO.iloc[0]
    dfIMO.drop(dfIMO.head(1).index, inplace=True)
    dfIMO.set_index('ID', inplace=True)
    
    #dfIMT = dfIMT.drop(columns=['Unit', 'Category', 'Channel','Image','Time'])
    dfIMT.columns = dfIMT.iloc[0]
    dfIMT.drop(dfIMT.head(1).index, inplace=True)
    dfIMT.set_index('ID', inplace=True)
    
    #dfIMTh = dfIMTh.drop(columns=['Unit', 'Category', 'Channel','Image','Time'])
    dfIMTh.columns = dfIMTh.iloc[0]
    dfIMTh.drop(dfIMTh.head(1).index, inplace=True)
    dfIMTh.set_index('ID', inplace=True)
    
    #dfIMF = dfIMF.drop(columns=['Unit', 'Category', 'Channel','Image','Time'])
    dfIMF.columns = dfIMF.iloc[0]
    dfIMF.drop(dfIMF.head(1).index, inplace=True)
    dfIMF.set_index('ID', inplace=True)
    
    # Need to transfer the information about the volume over the XYZ df without
    #. assumptions about df order
    for id in list(dfXYZ.index.values):
        # Get the volume associated with this ID from the volume df
        vol = dfVol['Volume'][id]
         

        # Filter out any surfaces with volume below threshold
        if (vol > volThresh):
            dfXYZ.at[id, 'Volume_um3'] = vol
            
            # First convert position from XYZ in microns to XYZ in voxels
            xPos = dfXYZ['Position X'][id]
            yPos = dfXYZ['Position Y'][id]
            zPos = dfXYZ['Position Z'][id]
    
            dfXYZ.at[id, 'Position X (voxels)'] = xPos/voxelWidth
            dfXYZ.at[id, 'Position Y (voxels)'] = yPos/voxelHeight
            dfXYZ.at[id, 'Position Z (voxels)'] = round(zPos/voxelDepth)
            
            
    
            # Get the ch1 mean intensity
            dfXYZ.at[id, 'uIntCh_1'] = dfIMO['Intensity Mean'][id]
            # Get the ch2 mean intensity
            dfXYZ.at[id, 'uIntCh_2'] = dfIMT['Intensity Mean'][id]
            # Get the ch3 mean intensity
            dfXYZ.at[id, 'uIntCh_3'] = dfIMTh['Intensity Mean'][id]
            # Get the ch4 mean intensity
            dfXYZ.at[id, 'uIntCh_4'] = dfIMF['Intensity Mean'][id]
        else:
            dfXYZ.drop(id, inplace = True)

    # Save the dataframe as a csv file    
    dfXYZ.to_csv(dirSV+file.split('.x')[0]+'.csv')